In [3]:
#Shared Memory
class SharedMemory:
    def __init__(self):
        self.data = {}

    def update(self, agent, value):
        self.data[agent] = value

    def get(self, agent):
        return self.data.get(agent, "")

    def get_all(self):
        return self.data

In [4]:
# NVIDIA LLM Call
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("NVIDIA_API_KEY")

def call_llm(prompt):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "meta/llama3-70b-instruct",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3
    }

    res = requests.post(url, headers=headers, json=data)
    return res.json()["choices"][0]["message"]["content"]

In [5]:
# Agents

In [6]:
#Search Agent
def search_agent(goal, memory):
    prompt = f"""
    You are a Search Agent.

    Goal: {goal}

    Find key EV market trends in India.

    Keep it short.
    """
    result = call_llm(prompt)
    memory.update("search", result)
    return result

In [7]:
#Analyst Agent
def analyst_agent(goal, memory):
    search_data = memory.get("search")

    prompt = f"""
    You are an Analyst.

    Based on:
    {search_data}

    Analyze trends and insights.
    """
    result = call_llm(prompt)
    memory.update("analyst", result)
    return result

In [8]:
#Writer Agent
def writer_agent(goal, memory):
    analysis = memory.get("analyst")

    prompt = f"""
    You are a Writer.

    Based on analysis:
    {analysis}

    Write a structured report.
    """
    result = call_llm(prompt)
    memory.update("writer", result)
    return result

In [9]:
#QA Agent
def qa_agent(goal, memory):
    report = memory.get("writer")

    prompt = f"""
    You are a QA Agent.

    Check this report:
    {report}

    Fix errors and improve clarity.
    """
    result = call_llm(prompt)
    memory.update("qa", result)
    return result

In [10]:
#Orchestrator (🔥 main logic)
def run_pipeline(goal):
    memory = SharedMemory()

    print("Running Search Agent...")
    search_agent(goal, memory)

    print("Running Analyst Agent...")
    analyst_agent(goal, memory)

    print("Running Writer Agent...")
    writer_agent(goal, memory)

    print("Running QA Agent...")
    qa_agent(goal, memory)

    return memory.get("qa")

In [11]:
#Run
goal = "Write a comprehensive market report on EV trends in India for 2025"

final_report = run_pipeline(goal)

print("\nFINAL REPORT:\n")
print(final_report)

Running Search Agent...
Running Analyst Agent...
Running Writer Agent...
Running QA Agent...

FINAL REPORT:

Here is the revised report with errors fixed and clarity improved:

**Comprehensive Market Report on EV Trends in India for 2025**

**Executive Summary**

The Indian electric vehicle (EV) market is poised for significant growth in 2025, driven by government support, rising demand, and increasing competition. The market is expected to reach 1.5 million units, with a growth rate of 40% year-over-year (YoY), making it an attractive opportunity for automakers, investors, and policymakers. This report provides an in-depth analysis of the key trends and insights shaping the Indian EV market, along with recommendations for stakeholders.

**Trend 1: Government Support**

The Indian government's goal of achieving 30% EV penetration by 2030 will drive growth in the EV market. Incentives such as subsidies, tax benefits, and investments in EV infrastructure will encourage adoption. Governme